In [ ]:
# P1-fix: os caminhos deste notebook são RELATIVOS a EXIST_VIDEOS/ (plano 11 §3).
# Se o kernel nascer noutro cwd, os asserts abaixo falham. Idempotente.
import os
from pathlib import Path

_TARGET = Path("/home/a67606/Thesis/EXIST_VIDEOS")
if Path.cwd() != _TARGET:
    assert _TARGET.is_dir(), f"diretório do projeto não encontrado: {_TARGET}"
    os.chdir(_TARGET)
assert Path("Dataset").is_dir(), f"cwd errado: {Path.cwd()} (sem Dataset/)"
assert Path("video_core.py").is_file(), "video_core.py não está no cwd"
assert Path("video_core_clip.py").is_file(), "video_core_clip.py não está no cwd"
print("cwd:", Path.cwd())

# ── Log em arquivo (independente de o Jupyter gravar outputs ou não) ────────
# O stdout é DUPLICADO: continua aparecendo na célula E vai para o arquivo.
# `_ORIG_STDOUT` guardado em globals() para re-executar esta célula não empilhar Tees.
# Arquivo DEDICADO ao braço CLIP — nunca sobrescrever analysis/probe_report.txt (B15).
import sys

_ORIG_STDOUT = globals().get("_ORIG_STDOUT", sys.stdout)


class _Tee:
    def __init__(self, path, term):
        self.f = open(path, "w", buffering=1, encoding="utf-8")
        self.term = term

    def write(self, s):
        self.term.write(s)
        self.f.write(s)
        return len(s)

    def flush(self):
        self.term.flush()
        self.f.flush()

    def isatty(self):
        return False


Path("analysis").mkdir(exist_ok=True)
LOG_PATH = "analysis/probe_report_clip.txt"
sys.stdout = _Tee(LOG_PATH, _ORIG_STDOUT)
print("[log] stdout duplicado para", LOG_PATH,
      "\n[log] stderr NAO e capturado — se houver traceback, copiar da tela")


# Sondas de contribuição do ramo de vídeo — checkpoint CLIP (`video_core_clip.py`)

**O que este notebook é.** Adaptação de `probe_video_contribution.ipynb` (o braço
VideoMAE, checkpoint `run_B2_bio_viewer_seed42_B15`) para o checkpoint novo que troca o
encoder de vídeo por CLIP ViT-Large/14 (`video_core_clip.py`,
`improvements/18_CLIP_VIDEO_BRANCH_SPEC.md`). As métricas agregadas (ICM-Norm) subiram
pouco (+0,001 a +0,02 em 5 de 6 métricas, uma caiu) — isso **não decide** se o vídeo passou
a carregar mais sinal causal. Regra do projeto (`video_core.py:1069`): métricas agregadas
não são evidência de contribuição de modalidade; só a sonda de permutação decide isso.

**Escopo desta sessão.** Só Stage A (forward + cache) + PORTÃO A (replay reproduz o
forward) + TESTE 1 (escada de permutação por modalidade) + TESTE 2 (`perm_across` /
`perm_within` / `noise_across` / `noise_within`) + comparação direta contra os números já
medidos no braço B15 (`improvements/16_VIDEO_INVESTIGATION_RESULTS.md`). Os TESTES 3–8 do
notebook original (escalas da fusão, diagnóstico do pool, sondas lineares, matriz de flips,
resumos, retorno de LoRA) **não são portados** — não respondem à pergunta desta sessão
("a dependência causal do vídeo mudou com o CLIP?") e vários dependem de estrutura que o
CLIP não tem (ver nota abaixo).

**O que este notebook NÃO é.** Evidência de que o vídeo ajuda. A sonda é **gatilho** para
gastar runs num braço treinado, nunca evidência. Quem prova é um braço treinado.

---

## O que muda de fato em relação ao original (resto é cópia fiel do método)

1. **Checkpoint** — `vcc.load_run_clip(RUN_DIR)` em vez de `vc.load_run(RUN_DIR,
   init_seed=42)`. `RUN_DIR` é uma constante no topo da célula 3, com placeholder —
   **AJUSTAR** para o nome real da pasta onde o checkpoint CLIP foi salvo.
2. **Dataset** — `vcc.EXISTVideoDatasetCLIP` em vez de `vc.EXISTVideoDataset`. Não é
   opcional: o `CLIPFrameEncoder` espera frames normalizados em mean/std do CLIP, não do
   ImageNet/VideoMAE, e `EXISTVideoDatasetCLIP` é quem faz essa renormalização afim
   (`video_core_clip.py:101-132`). Usar o dataset errado alimentaria o CLIP com estatística
   de normalização errada e o Stage A inteiro ficaria contaminado.
3. **`SLOTS`** — o slot de vídeo passa de `vc.VIDEO_DIM` (768) para `vcc.CLIP_VIDEO_DIM`
   (1024). Ordem das fatias preservada (vídeo, áudio, texto, bio) — o `forward` é herdado
   de `vc.EXISTVideoModel.forward`, então `torch.cat(parts, ...)` continua na mesma ordem.
   `fusion_in` esperado: `vcc.fusion_input_dim_clip(True) = 2944` (era 2688).
4. **Sem diagnóstico de pool no Stage A.** O hook original em `model.video_pool.attn`
   cacheava entropia/perfil temporal assumindo `last_hidden_state` com 1568 tokens
   espaço-temporais (VideoMAE: 8 posições × 196 patches). O `CLIPFrameEncoder` devolve
   **um CLS por frame** — `[B, 16, 1024]`, não `[B, 1568, 1024]`
   (`video_core_clip.py:198-239`) — então o pool aqui atende sobre 16 posições, não 1568.
   Esse diagnóstico só era consumido pelo TESTE 3 (não portado), então o hook de pool foi
   simplesmente removido do Stage A — não adaptado, porque nada nesta sessão o lê.
5. **PORTÃO C removido.** O original comparava contra dois números fixos medidos em
   `test.ipynb` para o checkpoint B15 (`task_b_icmnorm_hard` sob shuffle, seeds 0/1:
   0,4106 / 0,4316). Não existe equivalente para o checkpoint CLIP nesta sessão — não foi
   inventado. Só o PORTÃO A (replay reproduz as decisões do forward completo) e o PORTÃO B
   (métricas do forward batem com `final_metrics.json`) são portados; ambos continuam
   sendo os que decidem se os números **desta sessão** são válidos.

## Regra de leitura pré-registada (igual ao original, repetida aqui)

1. Todo arm é reportado como **média ± amplitude** sobre 5 derangements.
2. Uma diferença entre arms só é lida como diferença se **exceder a amplitude** dos
   próprios arms. Abaixo disso a leitura honesta é **inconclusivo a n=505**.
3. Na comparação final contra o B15: uma diferença ENTRE os dois deltas de permutação
   (`Δ_CLIP − Δ_B15`) só é lida como mudança real de dependência causal se exceder a
   **amplitude somada dos dois** (mesma disciplina, agora aplicada à diferença de
   diferenças). Abaixo disso: inconclusivo ao nível de ruído — não "sem mudança".


## 0. Imports, checkpoint CLIP, split congelado

In [ ]:
import json, math, warnings
from collections import namedtuple

import numpy as np
import pandas as pd
import torch

import video_core as vc
import video_core_clip as vcc
from transformers import XLMRobertaTokenizer, TrainingArguments

warnings.filterwarnings("ignore", category=UserWarning)

# ⚠ AJUSTAR: nome real da pasta onde o checkpoint a sondar foi salvo.
# Braço oficial (4 camadas, estado FINAL da época 15) — ver
# improvements/16_VIDEO_INVESTIGATION_RESULTS.md para o histórico de por que "CLIP_L2"
# (2 camadas, argmin) NÃO é mais o braço reportado.
RUN_DIR    = Path("run_B2_bio_viewer_seed42_CLIP")   # <<< AJUSTAR para o checkpoint real

# CACHE_DIR derivado de RUN_DIR.name (não mais fixo em "probe_cache_CLIP") — bug real
# encontrado numa rodada anterior desta sonda: um cache fixo sobrevive entre
# checkpoints diferentes e o Stage A é pulado ("forward saltado") mesmo sondando um
# checkpoint NOVO, silenciosamente medindo os pesos ANTIGOS. Cada RUN_DIR agora tem
# seu próprio cache; nunca mais precisa lembrar de apagar manualmente ao trocar de
# checkpoint.
CACHE_DIR  = Path("analysis") / f"probe_cache_{RUN_DIR.name}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = CACHE_DIR / "val_cache.pt"

N_DERANGEMENTS = 5
DERANGEMENT_SEEDS = list(range(N_DERANGEMENTS))     # 0..4

# Offsets dos slots — mesma ordem do original (video, audio, texto, bio), largura do
# slot de vídeo derivada de vcc.CLIP_VIDEO_DIM (1024) em vez de vc.VIDEO_DIM (768).
SLOTS = {
    "video": (0, vcc.CLIP_VIDEO_DIM),                                        # 0:1024
    "audio": (vcc.CLIP_VIDEO_DIM, vcc.CLIP_VIDEO_DIM + vc.AUDIO_PROJ_DIM),   # 1024:1792
    "texto": (vcc.CLIP_VIDEO_DIM + vc.AUDIO_PROJ_DIM,
              vcc.CLIP_VIDEO_DIM + vc.AUDIO_PROJ_DIM + vc.TEXT_DIM),         # 1792:2816
    "bio":   (vcc.CLIP_VIDEO_DIM + vc.AUDIO_PROJ_DIM + vc.TEXT_DIM,
              vcc.fusion_input_dim_clip(True)),                              # 2816:2944
}

METRIC_KEYS = [
    "task_a_icmnorm_hard", "task_a_icmsoftnorm_soft",
    "task_b_icmnorm_hard", "task_b_icmsoftnorm_soft",
    "task_c_icmnorm_hard", "task_c_icmsoftnorm_soft",
]

model, scaler = vcc.load_run_clip(RUN_DIR, init_seed=42)
with open(RUN_DIR / "meta.json") as f:
    META = json.load(f)
with open(RUN_DIR / "final_metrics.json") as f:
    FINAL = json.load(f)

print("\nmeta do checkpoint sondado:")
for k in ("video_encoder", "use_bio", "use_viewer_emb", "unfrozen_vision_layers",
          "video_hidden_dim", "n_frames", "frame_sampling", "fusion_in", "split_seed",
          "condition", "arm", "num_train_epochs", "selected_epoch", "epoch_selection"):
    print(f"  {k:22s} {META.get(k)}")
assert META["video_encoder"] == "clip", (
    "este checkpoint NAO e do braco CLIP (video_core_clip.load_run_clip ja teria "
    "abortado antes — se chegou aqui e falhou, e outra checagem).")
assert META["fusion_in"] == vcc.fusion_input_dim_clip(True) == 2944, "fusion_in inesperado"

tokenizer = XLMRobertaTokenizer.from_pretrained(vc.XLMR_NAME)
full_df = pd.read_json("Dataset/training/EXIST2026_training.json").transpose()
train_df, val_df = vc.build_frozen_split(full_df)
# vcc.EXISTVideoDatasetCLIP, NAO vc.EXISTVideoDataset — a renormalizacao afim para as
# estatisticas do CLIP acontece dentro dela (video_core_clip.py:101-132).
val_ds = vcc.EXISTVideoDatasetCLIP(val_df, tokenizer, scaler,
                                   vc.frames_dir("train"), vc.audio_dir("train"))
N_VAL = len(val_ds)
VAL_IDS = [str(i) for i in val_df["id_EXIST"].values]
print(f"\nval = {N_VAL} amostras (split congelado seed {vc.SPLIT_SEED})")
assert N_VAL == len(VAL_IDS) == 505, f"val tem {N_VAL}, esperado 505 (split congelado)"


## 1. Stage A — um forward sobre a val, cacheando o necessário

Captura, num só `trainer.predict`:

- **entrada da fusão** `[505, 2944]` (os quatro slots, no dtype exato do modelo);
- **logits** das três cabeças e os **soft labels** (`label_ids`, em ordem de dataset).

**Sem diagnóstico de pool nesta sessão** (ver nota 4 da célula introdutória): o hook
original em `model.video_pool.attn` assumia `last_hidden_state` de 1568 tokens
espaço-temporais; o `CLIPFrameEncoder` devolve um CLS por frame (`[B, 16, 1024]`), e nada
neste notebook (Stage A + PORTÃO A + TESTE 1 + TESTE 2) lê esse diagnóstico — só o TESTE 3
do original o consumia, e o TESTE 3 não é portado.

O hook em `model.fusion[0]` continua idêntico: captura a **entrada** da `Linear` de fusão,
o vetor concatenado como o modelo o fundiu, no dtype exato.

Idempotente **por `RUN_DIR`**: se `CACHE_DIR/val_cache.pt` existe, o forward é saltado.
`CACHE_DIR` é derivado de `RUN_DIR.name` (célula 3) — cada checkpoint tem seu próprio
cache, então trocar de `RUN_DIR` nunca reusa por engano o forward de outro checkpoint
(bug real encontrado e corrigido: um `CACHE_DIR` fixo sobrevivia entre checkpoints
diferentes e "forward saltado" media silenciosamente os pesos errados).


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", ("Stage A exige GPU. Este notebook roda na VM, não na máquina "
                          "local de desenvolvimento.")

args = TrainingArguments(output_dir="/tmp/probe_vid_clip", per_device_eval_batch_size=2,
                         bf16=True, remove_unused_columns=False, report_to="none",
                         label_names=["labels_a", "labels_b", "labels_c"])
trainer = vc.MultiTaskTrainer(model=model, args=args,
                              compute_metrics=vc.compute_metrics,
                              data_collator=vc.video_collate)
trainer.label_names = ["labels_a", "labels_b", "labels_c"]


def _run_stage_a():
    buf_fusion = []

    def hook_fusion(_m, inp, _out):
        buf_fusion.append(inp[0].detach().float().cpu())

    h1 = model.fusion[0].register_forward_hook(hook_fusion)
    try:
        out = trainer.predict(val_ds)
    finally:
        h1.remove()

    logits_a, logits_b, logits_c = vc._unpack_predictions(out.predictions)
    lab_a, lab_b, lab_c = out.label_ids

    cache = {
        "fusion_in": torch.cat(buf_fusion, dim=0),                       # [505, 2944]
        "logits_a": np.asarray(logits_a), "logits_b": np.asarray(logits_b),
        "logits_c": np.asarray(logits_c),
        "labels_a": np.asarray(lab_a), "labels_b": np.asarray(lab_b),
        "labels_c": np.asarray(lab_c),
        "ids": VAL_IDS,
    }
    torch.save(cache, CACHE_FILE)
    return cache


if CACHE_FILE.exists():
    cache = torch.load(CACHE_FILE, weights_only=False)
    print(f"[cache] carregado de {CACHE_FILE} (forward saltado)")
else:
    cache = _run_stage_a()
    print(f"[cache] gravado em {CACHE_FILE}")

FUSION_IN = cache["fusion_in"]
LAB_A, LAB_B, LAB_C = cache["labels_a"], cache["labels_b"], cache["labels_c"]
assert FUSION_IN.shape == (N_VAL, 2944), FUSION_IN.shape
assert cache["ids"] == VAL_IDS, (f"ordem do cache divergiu do val_df — cache antigo? "
                                 f"apagar {CACHE_DIR}/ e re-rodar")
print("shapes OK:", {k: tuple(v.shape) for k, v in cache.items()
                     if hasattr(v, "shape")})


## 2. PORTÃO — o replay reproduz o forward completo?

O cache já contém os **logits do forward real** (`trainer.predict`). Duas condições,
separadas para não confundir causas de falha:

| # | condição | o que significa se falhar |
|---|---|---|
| **A** | logits do replay == logits do forward | o replay está errado (dtype, batch, `train()`/`eval()`) — **aborta** |
| **B** | métricas do forward == `final_metrics.json` | esta sessão não reproduz a run de origem (ambiente/PyEvALL) — **avisa, não aborta** |

**Só a A é condição de validade deste notebook.** B compara com números medidos noutra
sessão; se divergir, os Δ *dentro* desta sessão continuam válidos (é Δ que decide as
hipóteses, não valor absoluto) — mas os absolutos não podem ser colados ao lado dos do
doc 16 sem a ressalva.

**Sem PORTÃO C nesta sessão** (ver nota 5 da célula introdutória) — não existe um
`test.ipynb` equivalente rodado sobre o checkpoint CLIP nesta investigação; não foram
inventados números de referência para preencher essa lacuna.

### `model.eval()` explícito

`nn.Module` nasce em `train()`. No caminho "cache carregado do disco" o modelo **nunca**
passa por `trainer.predict`, e o `Dropout` da fusão (`video_core.py:1049`, herdado sem
mudança pelo `EXISTVideoModelCLIP`) estaria ativo no replay.


In [ ]:
EvalPred = namedtuple("EvalPred", ["predictions", "label_ids"])

model.eval()
print("model.training =", model.training, "(tem de ser False)")
assert model.training is False

_FUSION_GPU = FUSION_IN.to(device)
REPLAY_BS, REPLAY_AUTOCAST = 2, True          # sobrescritos pela autocalibração abaixo


def replay(fusion_in_gpu, bs=None, use_autocast=None):
    """fusion -> heads. Modo (bs, autocast) vem da autocalibração desta célula."""
    bs = REPLAY_BS if bs is None else bs
    ac = REPLAY_AUTOCAST if use_autocast is None else use_autocast
    ctx = (torch.autocast("cuda", dtype=torch.bfloat16) if ac
           else torch.autocast("cuda", enabled=False))
    outs = ([], [], [])
    with torch.no_grad(), ctx:
        for i in range(0, fusion_in_gpu.shape[0], bs):
            fused = model.fusion(fusion_in_gpu[i:i + bs])
            outs[0].append(model.head_a(fused).squeeze(-1).float().cpu())
            outs[1].append(model.head_b(fused).float().cpu())
            outs[2].append(model.head_c(fused).float().cpu())
    return tuple(torch.cat(o).numpy() for o in outs)


def metrics_of(logits3):
    m = vc.compute_metrics(EvalPred(logits3, (LAB_A, LAB_B, LAB_C)))
    return {k: float(m[k]) for k in METRIC_KEYS}


def derangement(n, seed):
    """MESMA construção do original: permutação sem ponto fixo."""
    rs = np.random.RandomState(seed)
    while True:
        p = rs.permutation(n)
        if not (p == np.arange(n)).any():
            return p


def perturb(slot, index=None, replacement=None):
    """Cópia GPU de FUSION_IN com o slot substituído (permutação de linhas ou tensor)."""
    a, b = SLOTS[slot]
    x = FUSION_IN.clone()
    if index is not None:
        x[:, a:b] = FUSION_IN[np.asarray(index), a:b]
    if replacement is not None:
        x[:, a:b] = replacement.to(FUSION_IN.dtype)
    return x.to(device)


def delta_norm(slot, index=None, replacement=None):
    """||Dslot||2 médio por amostra — magnitude da perturbação realmente aplicada."""
    a, b = SLOTS[slot]
    base = FUSION_IN[:, a:b].float()
    new = base[np.asarray(index)] if index is not None else replacement.float()
    return float((new - base).norm(dim=1).mean())


# ================= autocalibração: qual modo reproduz o forward? =================
REF = (cache["logits_a"], cache["logits_b"], cache["logits_c"])
MODES = {"bs2+bf16": (2, True), "bs2+fp32": (2, False),
         "bs64+bf16": (64, True), "bs64+fp32": (64, False)}
cands = {}
print(f"\n{'modo':>10s}{'maxD logit':>14s}{'flips argmax B':>16s}{'flips A@0.5':>13s}")
for name, (bs, ac) in MODES.items():
    r = replay(_FUSION_GPU, bs=bs, use_autocast=ac)
    mx = max(float(np.abs(r[i] - REF[i]).max()) for i in range(3))
    fb = int((r[1].argmax(1) != REF[1].argmax(1)).sum())
    fa = int(((r[0] > 0) != (REF[0] > 0)).sum())
    cands[name] = (mx, fb, fa, bs, ac)
    print(f"{name:>10s}{mx:14.3e}{fb:16d}{fa:13d}")

best = min(cands, key=lambda k: (cands[k][1] + cands[k][2], cands[k][0]))
mx, fb, fa, REPLAY_BS, REPLAY_AUTOCAST = cands[best]
print(f"\nmodo escolhido: {best}  (bs={REPLAY_BS}, autocast_bf16={REPLAY_AUTOCAST})")

assert fb == 0 and fa == 0, (
    f"PORTAO A FALHOU: nenhum modo de replay reproduz as DECISOES do forward completo "
    f"(melhor: {best}, {fb} flips em B, {fa} em A, maxD logit {mx:.3e}). A equivalencia "
    f"fusion-in -> logits nao se sustenta nesta GPU/versao. Abandonar o replay: rodar os "
    f"testes 1/2 com forwards completos e manter o resto sobre o cache.")
if mx > 1e-3:
    print(f"[AVISO] maxD logit = {mx:.3e} sem flip de decisao. Replay valido para metricas "
          f"hard; tratar as metricas SOFT com reserva.")
print(">>> PORTAO A OK: o replay reproduz as decisoes do forward completo.")

# ================= B: esta sessao reproduz final_metrics.json? =================
m_forward = metrics_of(REF)          # métricas pelo MESMO código, sobre os logits reais
m_normal = metrics_of(replay(_FUSION_GPU))

print(f"\n{'metrica':30s}{'forward':>10s}{'replay':>10s}{'final_metrics':>15s}"
      f"{'D repl-fwd':>12s}{'D fwd-final':>13s}")
gate_replay, gate_session = True, True
for k in METRIC_KEYS:
    ref = float(FINAL[f"eval_{k}"])
    d1, d2 = m_normal[k] - m_forward[k], m_forward[k] - ref
    gate_replay &= abs(d1) < 1e-3
    gate_session &= abs(d2) < 1e-3
    print(f"{k:30s}{m_forward[k]:10.4f}{m_normal[k]:10.4f}{ref:15.4f}"
          f"{d1:+12.5f}{d2:+13.5f}")

assert gate_replay, (
    "PORTAO A FALHOU no nivel das METRICAS: os logits do replay reproduzem as decisoes "
    "mas as metricas divergem >1e-3. Isso so pode vir de nao-determinismo dentro do "
    "PyEvALL (mesma entrada, saida diferente) — investigar antes de seguir.")

SESSION_REPRODUCES = bool(gate_session)
print()
if SESSION_REPRODUCES:
    print(">>> PORTAO B OK: esta sessao reproduz a run de origem. Valores ABSOLUTOS "
          "comparaveis com o final_metrics.json do checkpoint CLIP.")
else:
    print("=" * 78)
    print("[AVISO ALTO] Esta sessao NAO reproduz os numeros da run de origem:")
    print("  - metricas do forward != final_metrics.json (coluna 'D fwd-final' acima)")
    print("  O replay esta VALIDO (portao A passou), logo todos os DELTAS deste notebook")
    print("  sao internamente consistentes e as hipoteses continuam decidiveis — elas")
    print("  dependem de comparacao ENTRE arms da mesma sessao, nao de absolutos.")
    print("  MAS: nao colar os valores absolutos ao lado dos do doc 16 sem esta ressalva,")
    print("  e investigar a divergencia (versao de PyEvALL? scaler? split?) antes de")
    print("  qualquer numero deste notebook ir para a tese.")
    print("=" * 78)


## 3. TESTE 2 — ruído pareado, 5 derangements, e permutação within-class

**Decide H3** (hipótese nula: a cabeça B é frágil a *qualquer* perturbação do vetor
fundido, e um Δ negativo não mede informação de vídeo).

Quatro arms, todos sobre o slot de vídeo (agora 1024-d, CLIP), com ‖Δ‖ **reportado** para
cada um:

| arm | o que é | serve para |
|---|---|---|
| `perm_across` | derangement livre (5 seeds) | a sonda original |
| `noise_across` | gaussiano isotrópico com ‖ε‖ = ‖Δ‖ de `perm_across` | H3 |
| `perm_within` | derangement **dentro da mesma classe gold de B** | H1 vs H3 |
| `noise_within` | gaussiano com ‖ε‖ = ‖Δ‖ de `perm_within` | controle de magnitude |

**Por que o arm within-class existe:** ruído isotrópico só é decisivo numa direção. Se
ruído ≥ permutação → H3 confirmada, limpo. Se ruído < permutação, é **ambíguo**:
"permutação é in-distribution-mas-errada" é uma explicação, "a cabeça é mais robusta a
direções OOD que a direções plausíveis" é outra, e nenhuma das duas fala de alinhamento
vídeo↔rótulo. O arm within-class fala: sob H3, within e across machucam igual; sob H1,
within machuca **menos**, porque o vetor trocado ainda carrega a assinatura de formato da
classe certa.

**Por que `noise_within` existe:** ‖Δ‖ de within-class deve ser menor que de across, e
"machuca menos" não vale nada se estiver confundido com "foi perturbado menos". O par
`perm_within` vs `noise_within` compara dano a magnitude casada.

**Regra pré-registada** (repetida da célula de abertura): só há diferença entre arms se
ela exceder a amplitude dos 5 derangements.

**CAVEAT do desenho** (declarado antes de ver qualquer número, igual ao original):
`perm_within` agrupa pelo GOLD de B, logo o arm é oracular **na construção**. É legítimo —
o gold escolhe o doador, nunca entra no modelo — mas tem de ser declarado antes que alguém
levante.


In [ ]:
LABELS_B = list(vc.LABELS_2_2)
mask_b = LAB_B[:, 0] != vc.SENTINEL_BC
gold_b = LAB_B[mask_b].argmax(axis=1)

rng = np.random.RandomState(20260809)
A_V, B_V = SLOTS["video"]
vid_base = FUSION_IN[:, A_V:B_V].float()

# ---- gold hard de B para o arm within-class (sentinela = grupo próprio) ----
group = np.full(N_VAL, -1, dtype=int)
group[mask_b] = gold_b
groups_desc = {int(g): int((group == g).sum()) for g in sorted(set(group.tolist()))}
print("grupos para permutação within-class (gold hard de B; -1 = sem voto válido):")
for g, n in groups_desc.items():
    name = "SEM_VOTO" if g == -1 else LABELS_B[g]
    print(f"  {name:>16s}  n={n}")


def derangement_within(seed):
    """Derangement dentro de cada grupo. Grupos de tamanho 1 ficam fixos (contados)."""
    rs = np.random.RandomState(1000 + seed)
    idx = np.arange(N_VAL)
    fixed = 0
    for g in set(group.tolist()):
        pos = np.where(group == g)[0]
        if len(pos) < 2:
            fixed += len(pos)
            continue
        for _ in range(500):
            p = rs.permutation(len(pos))
            if not (p == np.arange(len(pos))).any():
                break
        idx[pos] = pos[p]
        fixed += int((idx[pos] == pos).sum())
    return idx, fixed


def gaussian_at(target_norm, seed):
    """Ruído gaussiano com ‖ε‖₂ = target_norm em TODA amostra (média casada por construção)."""
    g = torch.from_numpy(np.random.RandomState(2000 + seed).randn(N_VAL, B_V - A_V).astype("float32"))
    g = g / g.norm(dim=1, keepdim=True) * target_norm
    return vid_base + g


arms = {}          # nome -> lista de (metrics, delta)
# perm_across
for s in DERANGEMENT_SEEDS:
    p = derangement(N_VAL, s)
    arms.setdefault("perm_across", []).append(
        (metrics_of(replay(perturb("video", index=p))), delta_norm("video", index=p)))
# perm_within
fixed_total = None
for s in DERANGEMENT_SEEDS:
    p, fixed = derangement_within(s)
    fixed_total = fixed
    arms.setdefault("perm_within", []).append(
        (metrics_of(replay(perturb("video", index=p))), delta_norm("video", index=p)))

d_across = float(np.mean([d for _, d in arms["perm_across"]]))
d_within = float(np.mean([d for _, d in arms["perm_within"]]))
print(f"\namostras sem par no grupo (ficaram fixas no arm within): {fixed_total}")
print(f"‖Δvid_feat‖ médio — across={d_across:.3f}  within={d_within:.3f}")

# noise arms, casados em magnitude
for s in DERANGEMENT_SEEDS:
    r = gaussian_at(d_across, s)
    arms.setdefault("noise_across", []).append(
        (metrics_of(replay(perturb("video", replacement=r))), delta_norm("video", replacement=r)))
    r = gaussian_at(d_within, 100 + s)
    arms.setdefault("noise_within", []).append(
        (metrics_of(replay(perturb("video", replacement=r))), delta_norm("video", replacement=r)))


def summarize(arm_name, key):
    vals = np.array([m[key] for m, _ in arms[arm_name]])
    base = m_normal[key]
    return vals.mean() - base, vals.max() - vals.min(), vals.mean()


print(f"\n{'arm':>14s}{'‖Δ‖':>9s}" + "".join(f"{k.replace('task_','').replace('icmnorm_','').replace('icmsoftnorm_',''):>16s}" for k in METRIC_KEYS))
print(f"{'normal':>14s}{0.0:9.3f}" + "".join(f"{m_normal[k]:16.4f}" for k in METRIC_KEYS))
for arm in ("perm_across", "noise_across", "perm_within", "noise_within"):
    dn = float(np.mean([d for _, d in arms[arm]]))
    row = f"{arm:>14s}{dn:9.3f}"
    for k in METRIC_KEYS:
        dmean, amp, _ = summarize(arm, k)
        row += f"{dmean:+11.4f}±{amp:.3f}"
    print(row)
print("\n(cada célula: Δ médio vs normal ± amplitude sobre os 5 derangements/seeds)")

KB = "task_b_icmnorm_hard"
dp, ap, _ = summarize("perm_across", KB)
dn_, an, _ = summarize("noise_across", KB)
dw, aw, _ = summarize("perm_within", KB)
dnw, anw, _ = summarize("noise_within", KB)
tol = max(ap, an, aw, anw)

print("\nLEITURA (Task B ICMNorm hard) — mesma arvore de leitura do notebook original")
print("(probe_video_contribution.ipynb, celula 5-continuacao), copiada literalmente.\n")
for arm, (dm, amp) in (("perm_across", (dp, ap)), ("noise_across", (dn_, an)),
                       ("perm_within", (dw, aw)), ("noise_within", (dnw, anw))):
    print(f"  {arm:14s} D={dm:+.4f} +-{amp:.4f}   ||Dvid_feat||="
          f"{float(np.mean([d for _, d in arms[arm]])):.3f}")
print(f"  tolerancia pre-registada = {tol:.4f}\n")

# --- discriminador PRIMARIO: within vs across, a magnitude praticamente casada ---
print(f"  [1] perm_within vs perm_across: |{dw:+.4f} - {dp:+.4f}| = {abs(dw - dp):.4f} "
      f"{'>' if abs(dw - dp) > tol else '<='} tol")
if abs(dw - dp) > tol and dw > dp:
    print("      REFUTA H3 na forma forte. As duas permutacoes tem ||D|| praticamente")
    print("      igual e a unica coisa que difere entre elas e a CLASSE GOLD DE B DO")
    print("      DOADOR — que o modelo nunca ve. Uma cabeca genericamente fragil nao")
    print("      pode dar dano ~zero sob uma perturbacao dessa magnitude.")
    print("      CONFIRMA o mecanismo da H1: o que o slot de video carrega e informacao")
    print("      de classe de B (registro/formato da fonte).")
    if abs(dw) < 0.01:
        print("      E MAIS: dano ~zero dentro da classe significa que o slot de video NAO")
        print("      carrega nada ALEM da classe de B. Se carregasse intensidade, categoria")
        print("      de C ou detalhe de enquadramento, trocar dentro da classe custaria")
        print("      algo. O ramo de video funciona como um classificador fraco de")
        print("      intencao da fonte, e nada mais. E uma afirmacao de TETO.")
elif abs(dw - dp) <= tol:
    print("      INCONCLUSIVO: trocar dentro da classe machuca o mesmo que entre classes.")
    print("      Compativel com H3 (fragilidade generica) — o slot nao carrega classe de B.")
else:
    print("      ANOMALIA nao prevista: within machuca MAIS que across. Reportar, nao ler.")

# --- controle de magnitude ---
print(f"\n  [2] perm_within vs noise_within (||D|| casada): |{dw:+.4f} - {dnw:+.4f}| = "
      f"{abs(dw - dnw):.4f} {'>' if abs(dw - dnw) > tol else '<='} tol")
if abs(dw - dnw) > tol:
    print("      Corrobora [1] a magnitude casada: ruido de mesma norma machuca, troca")
    print("      dentro da classe nao. O dano depende do CONTEUDO da perturbacao.")
else:
    print("      Abaixo da tolerancia — mesma direcao de [1], mas nao sustenta sozinho.")

# --- o teste original, agora no seu lugar: baixo poder ---
print(f"\n  [3] perm_across vs noise_across: "
      f"|{dp:+.4f} - {dn_:+.4f}| = {abs(dp - dn_):.4f} "
      f"{'>' if abs(dp - dn_) > tol else '<='} tol")
if abs(dp - dn_) <= tol:
    print("      SEM PODER a n=505. Isso NAO confirma H3 — 'indistinguivel' nao e")
    print("      'confirmado'. Ruido isotropico e o controle ERRADO; quem discrimina e")
    print("      permutacao condicional a classe com magnitude casada.")
else:
    print("      Separa: a permutacao machuca mais que ruido de mesma magnitude.")

print(f"\n  CAVEAT do desenho: perm_within agrupa pelo GOLD de B, logo o arm e oracular")
print(f"  na CONSTRUCAO. E legitimo — o gold escolhe o doador, nunca entra no modelo —")
print(f"  mas tem de ser declarado antes que alguem levante.")
print("\n  Nenhum destes numeros e evidencia de que o video AJUDA (video_core.py:1069).")


## 4. TESTE 1 — escada de permutação por modalidade

**Decide H1.** Mesma sonda, mesmas 5 seeds de derangement, aplicada a cada um dos quatro
slots. Os arms ficam comparáveis porque compartilham a permutação (mesma seed → mesmo
índice).

O que se lê: a **quota de dependência** por modalidade na mesma escala. Se permutar texto
derrubar a Task A dezenas de pontos e vídeo derrubar uma fração disso, o resultado do
vídeo deixa de ser anômalo — vira proporcional, e o achado é sobre o dataset, não sobre um
bug.


In [ ]:
ladder = {"video": arms["perm_across"]}     # já calculado no teste 2 — não recomputar
for slot in ("audio", "texto", "bio"):
    ladder[slot] = []
    for s in DERANGEMENT_SEEDS:
        p = derangement(N_VAL, s)
        ladder[slot].append((metrics_of(replay(perturb(slot, index=p))),
                             delta_norm(slot, index=p)))

print(f"{'slot':>8s}{'dim':>6s}{'‖Δ‖':>9s}" +
      "".join(f"{k.replace('task_','').replace('icmnorm_','H_').replace('icmsoftnorm_','S_'):>15s}" for k in METRIC_KEYS))
print(f"{'normal':>8s}{'':>6s}{0.0:9.3f}" + "".join(f"{m_normal[k]:15.4f}" for k in METRIC_KEYS))
for slot in ("video", "audio", "texto", "bio"):
    a, b = SLOTS[slot]
    dn = float(np.mean([d for _, d in ladder[slot]]))
    row = f"{slot:>8s}{b - a:6d}{dn:9.3f}"
    for k in METRIC_KEYS:
        vals = np.array([m[k] for m, _ in ladder[slot]])
        row += f"{vals.mean() - m_normal[k]:+10.4f}±{vals.max() - vals.min():.3f}"
    print(row)
print("\n(Δ médio vs normal ± amplitude sobre 5 derangements; ‖Δ‖ = perturbação aplicada ao slot)")

print("\nLEITURA:")
for k in ("task_a_icmnorm_hard", "task_b_icmnorm_hard", "task_c_icmnorm_hard"):
    deltas = {s: float(np.mean([m[k] for m, _ in ladder[s]])) - m_normal[k]
              for s in ("video", "audio", "texto", "bio")}
    order = sorted(deltas, key=lambda s: deltas[s])
    print(f"  {k:26s} hierarquia (mais nocivo → menos): " +
          " > ".join(f"{s}({deltas[s]:+.3f})" for s in order))
dv = float(np.mean([m["task_a_icmnorm_hard"] for m, _ in ladder["video"]])) - m_normal["task_a_icmnorm_hard"]
dt = float(np.mean([m["task_a_icmnorm_hard"] for m, _ in ladder["texto"]])) - m_normal["task_a_icmnorm_hard"]
print()
if dt < dv - 0.05:
    print("  CONFIRMA H1 (parte 1): o texto domina a Task A por larga margem. A")
    print("  contribuição pequena do vídeo em A é PROPORCIONAL, não anômala — o conteúdo")
    print("  sexista chega pela transcrição.")
else:
    print("  REFUTA H1 (parte 1): permutar texto NÃO domina a Task A. O modelo não está a")
    print("  ler conteúdo de nenhuma modalidade com força — o alvo passa a ser a fusão /")
    print("  as cabeças, não o encoder de vídeo.")
print("  ⚠ Estes Δ medem DEPENDÊNCIA do modelo treinado, não teto de informação da")
print("     modalidade. Um ramo que o treino aprendeu a ignorar dá Δ≈0 mesmo que a")
print("     modalidade contenha sinal.")


## 5. Comparação direta contra o braço B15 (VideoMAE)

Números do B15 copiados **literalmente** de
`improvements/16_VIDEO_INVESTIGATION_RESULTS.md` §3 (TESTE 1 e TESTE 2) — não
recomputados, não inferidos.

Regra: para cada métrica, calcula-se `Δ_CLIP − Δ_B15` (a diferença ENTRE os dois deltas de
permutação, não a diferença de métrica absoluta). Quando o B15 reportou amplitude
(‖derangements‖) para aquela métrica: se `|Δ_CLIP − Δ_B15|` exceder a **soma** das duas
amplitudes (a do CLIP desta sessão + a do B15 do doc 16), a dependência causal do vídeo
realmente mudou; caso contrário, inconclusivo ao nível de ruído — mesma disciplina do
doc 16, nenhuma regra nova inventada.

**Limitação declarada:** a tabela do TESTE 1 no doc 16 reporta só o Δ médio por slot, sem
amplitude (a amplitude só está publicada para o slot de vídeo, porque a linha `vídeo` do
TESTE 1 é literalmente a mesma linha que `perm_across` do TESTE 2 — mesmos números, mesma
run). Para os slots áudio/texto/bio do TESTE 1, portanto, a comparação abaixo é **só
descritiva** (mostra `Δ_CLIP − Δ_B15` sem veredito de tolerância) — não foi inventada uma
amplitude que o doc 16 não publicou.


In [ ]:
METRIC_LABEL = {
    "task_a_icmnorm_hard": "A_hard", "task_a_icmsoftnorm_soft": "A_soft",
    "task_b_icmnorm_hard": "B_hard", "task_b_icmsoftnorm_soft": "B_soft",
    "task_c_icmnorm_hard": "C_hard", "task_c_icmsoftnorm_soft": "C_soft",
}

# ---- referência B15, copiada literalmente do doc 16 §3 (TESTE 2, tabela 4 arms x 6 metricas) ----
B15_TEST2 = {
    "perm_across":  {"dnorm": 13.154,
                      "A_hard": (-0.0079, 0.009), "A_soft": (-0.0225, 0.014),
                      "B_hard": (-0.0631, 0.050), "B_soft": (-0.0527, 0.025),
                      "C_hard": (+0.0032, 0.015), "C_soft": (-0.0086, 0.019)},
    "noise_across": {"dnorm": 13.154,
                      "A_hard": (-0.0035, 0.039), "A_soft": (-0.0079, 0.025),
                      "B_hard": (-0.0353, 0.048), "B_soft": (-0.0136, 0.031),
                      "C_hard": (+0.0016, 0.019), "C_soft": (-0.0071, 0.019)},
    "perm_within":  {"dnorm": 13.119,
                      "A_hard": (+0.0152, 0.021), "A_soft": (+0.0012, 0.015),
                      "B_hard": (-0.0002, 0.021), "B_soft": (-0.0041, 0.008),
                      "C_hard": (+0.0210, 0.014), "C_soft": (-0.0045, 0.015)},
    "noise_within": {"dnorm": 13.119,
                      "A_hard": (-0.0231, 0.050), "A_soft": (-0.0138, 0.025),
                      "B_hard": (-0.0423, 0.046), "B_soft": (-0.0196, 0.026),
                      "C_hard": (-0.0071, 0.041), "C_soft": (-0.0071, 0.012)},
}

# ---- referência B15, TESTE 1 (escada por modalidade). O doc 16 só publica o Δ médio,
# SEM amplitude, exceto para "video" (= perm_across do TESTE 2, mesma linha). ----
B15_TEST1 = {
    "video": {"dim": 768, "dnorm": 13.154,
              "A_hard": -0.0079, "A_soft": -0.0225, "B_hard": -0.0631,
              "B_soft": -0.0527, "C_hard": +0.0032, "C_soft": -0.0086},
    "audio": {"dim": 768, "dnorm": 13.066,
              "A_hard": -0.0120, "A_soft": -0.0143, "B_hard": -0.0574,
              "B_soft": -0.0417, "C_hard": -0.0098, "C_soft": -0.0207},
    "texto": {"dim": 1024, "dnorm": 18.596,
              "A_hard": -0.2313, "A_soft": -0.1985, "B_hard": -0.2541,
              "B_soft": -0.1921, "C_hard": -0.2160, "C_soft": -0.1514},
    "bio":   {"dim": 128, "dnorm": 2.820,
              "A_hard": -0.0078, "A_soft": -0.0051, "B_hard": -0.0207,
              "B_soft": -0.0092, "C_hard": -0.0022, "C_soft": +0.0027},
}


def compare_row(label, delta_clip, amp_clip, delta_b15, amp_b15):
    if amp_b15 is None:
        diff = delta_clip - delta_b15
        print(f"  {label:36s} D_CLIP={delta_clip:+.4f}+-{amp_clip:.4f}  "
              f"D_B15={delta_b15:+.4f} (sem amplitude no doc 16)  "
              f"D_CLIP-D_B15={diff:+.4f}  [descritivo, sem veredito de tolerancia]")
        return
    diff = delta_clip - delta_b15
    tol = amp_clip + amp_b15
    if abs(diff) > tol:
        verdict = "MUDOU — excede a amplitude somada"
    else:
        verdict = "inconclusivo ao nivel de ruido (nao excede a amplitude somada)"
    print(f"  {label:36s} D_CLIP={delta_clip:+.4f}+-{amp_clip:.4f}  "
          f"D_B15={delta_b15:+.4f}+-{amp_b15:.4f}  D_CLIP-D_B15={diff:+.4f}  "
          f"tol={tol:.4f}  -> {verdict}")


print("=" * 90)
print("TESTE 2 — comparacao CLIP (esta sessao) vs B15 (doc 16 SS3), por arm x metrica")
print("=" * 90)
for arm in ("perm_across", "noise_across", "perm_within", "noise_within"):
    print(f"\narm = {arm}  (||D|| CLIP={float(np.mean([d for _, d in arms[arm]])):.3f}  "
          f"||D|| B15={B15_TEST2[arm]['dnorm']:.3f})")
    for k in METRIC_KEYS:
        lbl = METRIC_LABEL[k]
        d_clip, amp_clip, _ = summarize(arm, k)
        d_b15, amp_b15 = B15_TEST2[arm][lbl]
        compare_row(lbl, d_clip, amp_clip, d_b15, amp_b15)

print()
print("=" * 90)
print("TESTE 1 — comparacao CLIP (esta sessao) vs B15 (doc 16 SS3), por slot x metrica")
print("=" * 90)
for slot in ("video", "audio", "texto", "bio"):
    ref = B15_TEST1[slot]
    dn_clip = float(np.mean([d for _, d in ladder[slot]]))
    print(f"\nslot = {slot}  (dim CLIP={SLOTS[slot][1] - SLOTS[slot][0]}, "
          f"dim B15={ref['dim']}  ||D|| CLIP={dn_clip:.3f}  ||D|| B15={ref['dnorm']:.3f})")
    for k in METRIC_KEYS:
        lbl = METRIC_LABEL[k]
        vals = np.array([m[k] for m, _ in ladder[slot]])
        d_clip = float(vals.mean() - m_normal[k])
        amp_clip = float(vals.max() - vals.min())
        d_b15 = ref[lbl]
        # amplitude do B15 só existe para o slot "video" (= perm_across do TESTE 2).
        amp_b15 = B15_TEST2["perm_across"][lbl][1] if slot == "video" else None
        compare_row(lbl, d_clip, amp_clip, d_b15, amp_b15)

print()
print("=" * 90)
print("LEITURA GERAL")
print("=" * 90)
print("Regra: uma diferenca so e lida como mudanca REAL de dependencia causal do video se")
print("|D_CLIP - D_B15| exceder a amplitude SOMADA dos dois. Abaixo disso: inconclusivo ao")
print("nivel de ruido, nao 'sem mudanca' — mesma disciplina pre-registrada do doc 16, nao")
print("uma regra nova. Nenhum numero aqui e evidencia de que o video AJUDA")
print("(video_core.py:1069) — decide apenas se a DEPENDENCIA causal medida pela sonda")
print("mudou entre o encoder VideoMAE (B15) e o encoder CLIP (esta sessao).")
